In [ ]:
import subprocess
import requests
from pathlib import Path

print("WEEK 2 CONTAINER & SERVICE HEALTH CHECK")
print("=" * 50)

# Find project root
current_dir = Path.cwd()
if current_dir.name == "services" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "docker-compose.yml").exists():
    project_root = current_dir
else:
    print("✗ Could not find project root")
    exit()

print(f"Project root: {project_root}")

# Step 1: Check if containers are built and running
print("\n1. Checking container status...")
try:
    result = subprocess.run(
        ["docker", "compose", "ps", "--format", "table"],
        cwd=str(project_root),
        capture_output=True,
        text=True,
        timeout=10
    )
    
    if result.returncode == 0 and result.stdout.strip():
        print("✓ Containers are running:")
        for line in result.stdout.strip().split('\n'):
            print(f"   {line}")
    else:
        print("✗ No containers running or docker compose failed")
        print("Please run the build commands from the markdown cell above")
        exit()
        
except Exception as e:
    print(f"✗ Error checking containers: {e}")
    print("Please run the build commands from the markdown cell above")
    exit()

# Step 2: Check all service health (corrected endpoints)
print("\n2. Checking service health...")
services_to_test = {
    "FastAPI": "http://localhost:8000/api/v1/health",
    "PostgreSQL (via API)": "http://localhost:8000/api/v1/health", 
    "Ollama": "http://localhost:11434/api/version",
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Airflow": "http://localhost:8080/health"
}

all_healthy = True
for service_name, url in services_to_test.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"✓ {service_name}: Healthy")
        else:
            print(f"✗ {service_name}: HTTP {response.status_code}")
            all_healthy = False
    except requests.exceptions.ConnectionError:
        print(f"✗ {service_name}: Not accessible")
        all_healthy = False
    except Exception as e:
        print(f"✗ {service_name}: {type(e).__name__}")
        all_healthy = False

print("\n" + "=" * 50)
if all_healthy:
    print("✓ ALL SERVICES HEALTHY! Ready for Week 2 development.")
else:
    print("✗ Some services need attention.")

WEEK 2 CONTAINER & SERVICE HEALTH CHECK
Project root: /Users/kumarrohit/Desktop/ArXiv Paper Curator/Arxiv_Paper_Curator

1. Checking container status...
✓ Containers are running:
   NAME                    IMAGE                                            COMMAND                  SERVICE                 CREATED             STATUS                                 PORTS
   rag-airflow             arxiv_paper_curator-airflow                      "/entrypoint.sh"         airflow                 About an hour ago   Up About an hour (healthy)             0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
   rag-api                 arxiv_paper_curator-api                          "uvicorn src.main:ap…"   api                     43 minutes ago      Up 43 minutes (healthy)                0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
   rag-clickhouse          clickhouse/clickhouse-server:24.8-alpine         "/entrypoint.sh"         clickhouse              About an hour ago   Up About an hour (healthy)    